In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score

# 실제 정답: 환자 5명(1), 정상 95명(0)
y_true = np.array([1]*5 + [0]*95)

# 모델 A: 전원 '정상'이라고 판정한 엉터리 모델
y_pred_A = np.zeros(100, dtype=int)

# 모델 B: 10명을 환자로 지목, 그중 4명이 실제 환자
y_pred_B = np.array([1,1,1,1,0] + [1]*6 + [0]*89)

for name, y_pred in [('모델 A', y_pred_A), ('모델 B', y_pred_B)]:
    print(f'--- {name} ---')
    print('혼동 행렬:')
    print(confusion_matrix(y_true, y_pred))
    print(f'정확도 : {accuracy_score(y_true, y_pred):.2f}')
    print(f'정밀도 : {precision_score(y_true, y_pred, zero_division=0):.2f}')
    print(f'재현율 : {recall_score(y_true, y_pred, zero_division=0):.2f}')
    print(f'F1 점수: {f1_score(y_true, y_pred, zero_division=0):.2f}\n')

--- 모델 A ---
혼동 행렬:
[[95  0]
 [ 5  0]]
정확도 : 0.95
정밀도 : 0.00
재현율 : 0.00
F1 점수: 0.00

--- 모델 B ---
혼동 행렬:
[[89  6]
 [ 1  4]]
정확도 : 0.93
정밀도 : 0.40
재현율 : 0.80
F1 점수: 0.53



In [ ]:
# 📌 III-8의 모델 다시 만들기(이 코드 셀은 책의 III-10 단원에 실려 있지 않음)
# 다음 코드 셀의 예제 2는 III-8 단원에서 학습시킨 모델을 평가하는 코드이기 때문에,
# III-8과 똑같은 모델을 여기에서 그대로 실행시킴.

import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Flatten
from tensorflow.keras.utils import to_categorical

# MNIST 데이터셋 불러오기
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# 데이터 정규화
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 레이블 데이터를 원-핫 인코딩으로 변환
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

# 소프트맥스 회귀 모델 구축
model = Sequential([
    Input(shape=(28, 28)),
    Flatten(),    # 2차원(28x28)을 1차원(784)으로 평탄화
    Dense(10, activation='softmax')
])

# 모델 컴파일
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# 학습
model.fit(x_train, y_train_cat, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# 모델 평가
test_loss, test_acc = model.evaluate(x_test, y_test_cat)
print(f'\n모델 정확도: {test_acc*100:.2f} %\n')

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8677 - loss: 0.5125 - val_accuracy: 0.9147 - val_loss: 0.3167
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9114 - loss: 0.3181 - val_accuracy: 0.9210 - val_loss: 0.2870
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9178 - loss: 0.2934 - val_accuracy: 0.9233 - val_loss: 0.2773
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9216 - loss: 0.2813 - val_accuracy: 0.9262 - val_loss: 0.2686
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9234 - loss: 0.2738 - val_accuracy: 0.9257 - val_loss: 0.2685
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9256 - loss: 0.2682 - val_accuracy: 0.9264 - val_loss: 0.2650
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9272 - loss: 0.2633 - val_accuracy: 0.9283 - val_loss: 0.2622
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# 위 코드 셀에서 학습시킨 모델과 테스트 데이터 사용
y_pred = np.argmax(model.predict(x_test), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

# 10x10 혼동 행렬
cm = confusion_matrix(y_true, y_pred)
print('\n[ 혼동 행렬 ]')
print(cm)

# 클래스별 정밀도, 재현율, F1 점수
print('\n\n[ 클래스별 성능 ]')
print(classification_report(y_true, y_pred, digits=3))

# 가장 많이 혼동한 조합 5개 찾기
print('\n[ 가장 많이 혼동한 조합 5개 ]')
np.fill_diagonal(cm, 0)          # 맞힌 경우(대각선)는 제외
for _ in range(5):
    i, j = np.unravel_index(cm.argmax(), cm.shape)
    print(f'실제 {i} → 예측 {j} : {cm[i, j]}건')
    cm[i, j] = 0

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 759us/step

[ 혼동 행렬 ]
[[ 960    0    1    2    0    6    8    2    1    0]
 [   0 1117    3    2    0    1    4    2    6    0]
 [   5    8  934   16    4    3   13    8   37    4]
 [   3    0   21  917    0   23    3    9   25    9]
 [   1    2   10    1  892    0   12    5   10   49]
 [  10    3    5   30    6  771   17    6   37    7]
 [  11    3    7    1    6    8  919    0    3    0]
 [   1    6   29    5    5    1    0  937    3   41]
 [   7    9    8   17    7   19   10    6  886    5]
 [  11    7    1   12   17    4    0   11   11  935]]


[ 클래스별 성능 ]
              precision    recall  f1-score   support

           0      0.951     0.980     0.965       980
           1      0.967     0.984     0.976      1135
           2      0.917     0.905     0.911      1032
           3      0.914     0.908     0.911      1010
           4      0.952     0.908     0.930       982
           5      0.922     0.864     0.892       892
           6      0.93